In [122]:
import pandas as pd
df = pd.read_csv("C:\\Users\\anilr\\Downloads\\Customer_datasets - Customer_datasets.csv")
df.head(10)

,Customer_ID,Name,Age,Salary,Join_Date,Department,Performance_Score,Email,City,Gender,Is_Active
0,1357,Customer_1357,27,29664.12,2022-06-14,HR,69.1,user1356@outlook.com,Houston,Male,FALSE
1,985,Customer_985,27,47373.17,NaN,Marketing,1.9,user984@yahoo.com,Dallas,Female,FALSE
2,1828,Customer_860,62,63380.36,20170530,Operations,78.2,user859@gmail.com,NaN,Male,NaN
3,1985,Customer_1985,thirty,56746.57,2016-12-19 00:00:00,IT,85.8,user1984@company.com,New York,Male,TRUE
4,1294,Customer_1294,41,61654.21,2019-03-12,Finance,82.5,user1293@gmail.com,NEW YORK,Other,TRUE
5,1470,Customer_1470,54,58544.89,2022-02-24,Finance,48.7,user1469@outlook.com,San Antonio,Male,FALSE
6,1211,Customer_1211,31,61379.54,01-13-2020,Marketing,NaN,user1210@gmail.com,Los Angeles,Female,TRUE
7,368,Customer_368,65,90205.71,2021-06-18,IT,72.4,user367@company.com,San Diego,Male,TRUE
8,1704,NaN,71,77165.23,NaN,Marketing,37.6,user1703@gmail.com,Chicago,Other,TRUE
9,1384,Customer_1384,61,32664.99,2021-07-11,Sales,48,user1383@gmail.com,Houston,Other,TRUE


# Basic Exploration & Missing Values


In [123]:
# 1. How many missing values are there in each column? Which column has the highest number of missing values?
df.isna().sum()

Customer_ID            0
Name                 145
Age                   84
Salary                97
Join_Date            254
Department            51
Performance_Score     39
Email                 49
City                  43
Gender                34
Is_Active             25
dtype: int64

In [124]:
# 2. What percentage of rows have at least one missing value?
(df.isna().sum() / len(df)) * 100

Customer_ID           0.000000
Name                  7.142857
Age                   4.137931
Salary                4.778325
Join_Date            12.512315
Department            2.512315
Performance_Score     1.921182
Email                 2.413793
City                  2.118227
Gender                1.674877
Is_Active             1.231527
dtype: float64

In [125]:
# 3. Drop all rows that have more than 3 missing values. How many rows remain?
df = df.dropna()
df.isna().sum()
df.head(10)

,Customer_ID,Name,Age,Salary,Join_Date,Department,Performance_Score,Email,City,Gender,Is_Active
0,1357,Customer_1357,27,29664.12,2022-06-14,HR,69.1,user1356@outlook.com,Houston,Male,FALSE
3,1985,Customer_1985,thirty,56746.57,2016-12-19 00:00:00,IT,85.8,user1984@company.com,New York,Male,TRUE
4,1294,Customer_1294,41,61654.21,2019-03-12,Finance,82.5,user1293@gmail.com,NEW YORK,Other,TRUE
5,1470,Customer_1470,54,58544.89,2022-02-24,Finance,48.7,user1469@outlook.com,San Antonio,Male,FALSE
7,368,Customer_368,65,90205.71,2021-06-18,IT,72.4,user367@company.com,San Diego,Male,TRUE
9,1384,Customer_1384,61,32664.99,2021-07-11,Sales,48,user1383@gmail.com,Houston,Other,TRUE
10,1788,Customer_1788,26,46429.39,2022-12-12,HR,27.2,user1787@outlook.com,San Antonio,Other,FALSE
13,383,Customer_383,37,42660.68,2018-11-26,Sales,3.6,user382@outlook.com,Houston TX,Male,FALSE
14,354,Customer_354,47,22214.34,"June 13, 2015",Marketing,46,user353@gmail.com,San Jose,Other,FALSE
15,619,Customer_619,70,38895.56,2022-05-20,Finance,99.5,user618@yahoo.com,Los Angeles,Male,TRUE


# Duplicates

In [126]:
# 4. How many exact duplicate rows are present in the dataset?
(df.duplicated().sum().sum() / len(df)) * 100

np.float64(1.5128593040847202)

In [127]:
# 5. How many duplicate Customer_IDs are there? Keep only the first occurrence of each Customer_ID.
df['Customer_ID'].dropna().sum()
df['Customer_ID']

0       1357
3       1985
4       1294
5       1470
7        368
        ... 
2024    1096
2025    1131
2026    1295
2028    1460
2029    1127
Name: Customer_ID, Length: 1322, dtype: int64

# Age Column

In [128]:
# 6. Convert the Age column to numeric. How many values could not be converted (became NaN)?
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Age']

0       27.0
3        NaN
4       41.0
5       54.0
7       65.0
        ... 
2024    74.0
2025    74.0
2026    72.0
2028    69.0
2029    65.0
Name: Age, Length: 1322, dtype: float64

In [129]:
# 7. After converting Age to numeric, identify and remove outliers using the IQR method. How many outliers were removed?
Q1 = df['Age'].quantile(0.25)
Q2 = df['Age'].quantile(0.75)
IQR = Q2-Q1
lower_bond = Q1 - 1.5 * IQR
upper_bond = Q1 + 1.5 * IQR

outliers = df[
    (df["Age"] < lower_bond) |
    (df["Age"] > upper_bond)]
print(len(outliers))
print(outliers[['Age']])

6
        Age
240   999.0
870   200.0
1221  150.0
1375  200.0
1485  999.0
1902  999.0


In [130]:
# 8. Replace remaining missing values in Age with the median age.
df['Age'] = df['Age'].fillna(df['Age']).median()
df['Age']

0       50.0
3       50.0
4       50.0
5       50.0
7       50.0
        ... 
2024    50.0
2025    50.0
2026    50.0
2028    50.0
2029    50.0
Name: Age, Length: 1322, dtype: float64

# Salary Column

In [131]:
# 9. Clean the Salary column: remove currency symbols ($), commas, and convert everything possible to numeric. How many values remained non-numeric?
salary = df["Salary"].astype(str)
cleaned_salary = (salary.str.replace("$", "", regex=False)
                                                        .str.replace(",", "", regex=False)
)

numeric_salary = pd.to_numeric(cleaned_salary, errors="coerce")

non_numeric = numeric_salary.isna() & salary.notna()

print("Non-numeric values:", non_numeric.sum())

Non-numeric values: 19


In [132]:
# 10. After cleaning, find the mean and median salary. Which one is more affected by outliers?
df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')

df_mean = (df['Salary']).median()
print(df_mean)

Q1 = df['Salary'].quantile(0.25)
Q2 = df['Salary'].quantile(0.75)
IQR = Q2-Q1
lower_bond = Q1 - 1.5 * IQR
upper_bond = Q1 + 1.5 * IQR

outliers = df[
    (df["Salary"] < lower_bond) |
    (df["Salary"] > upper_bond)]
print(len(outliers))
print(outliers[['Salary']])

df['Salary']


55381.4
147
        Salary
7     90205.71
32    80881.24
49    86641.67
64    82975.05
96    76319.24
...        ...
1960  76664.67
1971      0.00
1972  77762.80
1984  15528.52
1991  81485.64

[147 rows x 1 columns]


0       29664.12
3       56746.57
4       61654.21
5       58544.89
7       90205.71
          ...   
2024    43331.22
2025    46818.31
2026    64255.45
2028    46179.42
2029    58869.43
Name: Salary, Length: 1322, dtype: float64

# Date Column

In [133]:
# 11. Convert the Join_Date column to proper datetime format. How many dates failed to parse?
df['Join_Date'] = pd.to_datetime(df['Join_Date'],dayfirst=True,errors='coerce')
df['Join_Date'].value_counts()
df['Join_Date'] = df['Join_Date'].ffill()
df['Join_Date'].isna().sum()

C:\Users\anilr\AppData\Local\Temp\ipykernel_3172\89928360.py:2: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Join_Date'] = pd.to_datetime(df['Join_Date'],dayfirst=True,errors='coerce')


np.int64(0)

In [134]:
# 12. Extract the year from the cleaned Join_Date. What is the most common joining year?
df['Join_Year'] = df['Join_Date'].dt.year
print(f"common joining year:", df['Join_Year'].mode()[0])
df['Join_Year'].head()

common joining year: 2019


0    2022
3    2022
4    2019
5    2022
7    2021
Name: Join_Year, dtype: int32

# Categorical Columns (Inconsistencies)

In [135]:
# 13. Standardize the Department column (fix casing, trim spaces, fix common typos like
#“Sale” → “Sales”, “Financ” → “Finance”, etc.). How many unique departments remain after cleaning?
df['Department'] = df['Department'].replace({"H R":"HR",
                                 "Financ":"Finance",
                                 "Sale":"Sales",
                                 "SALES":"Sales",
                                 "sales":"Sales",
                                "Marketing Dept":"Marketing"

                                })
df['Department'].unique()

array(['HR', 'IT', 'Finance', 'Sales', 'Marketing', 'Operations',
       'Support', 'Unknown'], dtype=object)

In [137]:
# 14. Clean the City column (standardize names, fix “Los Angles” → “Los Angeles”, “NY” →
# “New York”, “Philly” → “Philadelphia”, etc.). List the final unique cities.
df['City'] = df['City'].replace({"Houston TX":"Houston",
                                 "NEW YORK":"New York",
                                 "new york":"New York",
                                 "Los Angles":"Los Angeles",
                                 "Philadelphia":"Philly",
                                "NY":"New York"

                                })
df['City'].unique()

array(['Houston', 'New York', 'San Antonio', 'San Diego', 'San Jose',
       'Los Angeles', 'Chicago', 'Philly', 'Phoenix', 'Dallas', 'Unknown'],
      dtype=object)

In [138]:
# 15. Standardize the Gender column so it only contains “Male”, “Female”, and “Other”. How many records were changed?
df['Gender'] = df['Gender'].replace({"m":"Male",
                                 "M":"Male",
                                 "male":"Male",
                                 "female":"Female",
                                 "F":"Female",
                                "f":"Female",
                                   "Unknown":"Other"

                                })
df['Gender'].unique()

array(['Male', 'Other', 'Female'], dtype=object)

# Performance_Score & Is_Active

In [147]:
# 16. Convert Performance_Score to numeric. Replace non-numeric values with NaN, then fill missing scores with the mean.
df['Performance_Score'] = pd.to_numeric(df['Performance_Score'], errors='coerce')
df['Performance_Score'] = df['Performance_Score'].fillna(df['Performance_Score'].mean())
df['Performance_Score'] 

0       69.100000
3       85.800000
4       82.500000
5       48.700000
7       72.400000
          ...    
2024    92.200000
2025    46.800000
2026    52.804633
2028    17.500000
2029    47.400000
Name: Performance_Score, Length: 1322, dtype: float64

In [161]:
# 17. Clean the Is_Active column so it becomes a proper boolean (True/False). Map values
# like “Yes”, “Y”, “1”, “Active”, “true” → True and “No”, “N”, “0”, “Inactive”, “false” → False.
# Convert Is_Active values to True/False

active_map = {
    'Yes': True,
    'Y': True,
    '1': True,
    'Active': True,
    'true': True,
    
    'No': False,
    'N': False,
    '0': False,
    'Inactive': False,
    'false': False
}

df['Is_Active'] = df['Is_Active'].map(active_map)
df['Is_Active'].unique()

array([nan], dtype=object)

# Email & Name


In [166]:
# 18. Flag invalid email addresses (those that do not contain “@” or have incomplete domains). How many invalid emails are there? 
# Flag invalid email addresses

df['Invalid_Email'] = ~df['Email'].astype('string').str.match(
    r'^[^@\s]+@[^@\s]+\.[^@\s]+$',
    na=False
)

# Count invalid emails
invalid_count = df['Invalid_Email'].sum()

print("Number of invalid emails:", invalid_count)

df.loc[df['Invalid_Email'], ['Email', 'Invalid_Email']]

Number of invalid emails: 16


,Email,Invalid_Email
205,user@domain,True
271,user@.com,True
339,user@.com,True
453,user@domain,True
463,noemail,True
742,user@domain,True
895,user@.com,True
1505,user@domain,True
1652,user@.com,True
1713,invalid@,True


In [170]:
# 19. Clean the Name column: replace empty strings, “N/A”, and “Unknown” with NaN. How many names are still missing after this step?
import numpy as np
df['Name'] = df['Name'].replace(r'^\s*$', np.nan, regex=True)
df['Name'] = df['Name'].replace(['N/A', 'Unknown'], np.nan)

print(df['Name'].isna().sum())

7


In [136]:
df.columns


Index(['Customer_ID', 'Name', 'Age', 'Salary', 'Join_Date', 'Department',
       'Performance_Score', 'Email', 'City', 'Gender', 'Is_Active',
       'Join_Year'],
      dtype='object')

# Final Challenge


In [174]:
# 20. Perform a complete data cleaning pipeline on the entire dataset and save the cleaned version as cleaned_dataset.csv.
# Report the final shape of the cleaned dataset and briefly describe the main cleaning steps you applied.
df.to_csv("Anil_Customer_datasets")
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1322 entries, 0 to 2029
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Customer_ID        1322 non-null   int64         
 1   Name               1315 non-null   object        
 2   Age                1322 non-null   float64       
 3   Salary             1287 non-null   float64       
 4   Join_Date          1322 non-null   datetime64[ns]
 5   Department         1322 non-null   object        
 6   Performance_Score  1322 non-null   float64       
 7   Email              1322 non-null   object        
 8   City               1322 non-null   object        
 9   Gender             1322 non-null   object        
 10  Is_Active          0 non-null      object        
 11  Join_Year          1322 non-null   int32         
 12  Invalid_Email      1322 non-null   boolean       
dtypes: boolean(1), datetime64[ns](1), float64(3), int32(1), int64(1), ob